# SGC vs GCN Benchmark Harness

Runs both models on Cora / Citeseer / Pubmed across multiple seeds and produces a comparison table.

**Outputs per run:** test acc, val acc, best epoch, wall-clock training time, full loss/accuracy curves.

**Outputs aggregated:** mean ± std test accuracy and training time per (model, dataset).

## 1. Install dependencies

In [26]:
%%capture
!pip install torch torchvision torchaudio
!pip install torch_geometric

## 2. Imports and models

In [27]:
import time
from dataclasses import dataclass, field
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import torch_geometric.transforms as T
from torch_geometric.utils import add_self_loops, degree

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [28]:
class SGC(nn.Module):
    """Single linear layer applied to precomputed S^K X."""
    def __init__(self, nfeat, nclass):
        super().__init__()
        self.lin = nn.Linear(nfeat, nclass)
    def forward(self, x):
        return self.lin(x)

class GCN(nn.Module):
    """Standard 2-layer GCN, Kipf & Welling baseline."""
    def __init__(self, nfeat, nhid, nclass, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(nfeat, nhid, cached=True)
        self.conv2 = GCNConv(nhid, nclass, cached=True)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)

## 3. SGC feature precompute and result container

In [29]:
def sgc_precompute(data, K=2):
    """Compute S^K X. Returns (features, precompute_time_seconds)."""
    t0 = time.perf_counter()
    edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
    row, col = edge_index
    deg = degree(col, data.num_nodes, dtype=data.x.dtype)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
    edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    adj = torch.sparse_coo_tensor(edge_index, edge_weight,
                                   (data.num_nodes, data.num_nodes)).to(data.x.device)
    x = data.x
    for _ in range(K):
        x = torch.spmm(adj, x)
    if x.is_cuda: torch.cuda.synchronize()
    return x, time.perf_counter() - t0

@dataclass
class RunResult:
    model: str
    dataset: str
    seed: int
    test_acc: float
    val_acc: float
    best_epoch: int
    train_time: float
    train_loss_curve: List[float] = field(default_factory=list)
    val_loss_curve: List[float] = field(default_factory=list)
    train_acc_curve: List[float] = field(default_factory=list)
    val_acc_curve: List[float] = field(default_factory=list)
    test_acc_curve: List[float] = field(default_factory=list)

def _accuracy(logits, y, mask):
    return (logits[mask].argmax(dim=1) == y[mask]).float().mean().item()

## 4. Train/eval routines (one per model)

In [30]:
def run_sgc(data, num_classes, seed, K=2, lr=0.2, weight_decay=5e-6, epochs=100):
    torch.manual_seed(seed); np.random.seed(seed)
    x_prop, t_pre = sgc_precompute(data, K=K)
    model = SGC(x_prop.shape[1], num_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    tr_l, va_l, tr_a, va_a, te_a = [], [], [], [], []
    best_va, best_te, best_ep = 0.0, 0.0, 0
    t0 = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        out = model(x_prop)
        loss = crit(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out = model(x_prop)
            tr_l.append(crit(out[data.train_mask], data.y[data.train_mask]).item())
            va_l.append(crit(out[data.val_mask], data.y[data.val_mask]).item())
            tr_a.append(_accuracy(out, data.y, data.train_mask))
            va_a.append(_accuracy(out, data.y, data.val_mask))
            te_a.append(_accuracy(out, data.y, data.test_mask))
        if va_a[-1] > best_va:
            best_va, best_te, best_ep = va_a[-1], te_a[-1], epoch
    if device.type == 'cuda': torch.cuda.synchronize()
    return RunResult('SGC', '', seed, best_te, best_va, best_ep,
                     t_pre + (time.perf_counter() - t0),
                     tr_l, va_l, tr_a, va_a, te_a)

def run_gcn(data, num_classes, seed, nhid=16, lr=0.01, weight_decay=5e-4,
            dropout=0.5, epochs=200, patience=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = GCN(data.num_features, nhid, num_classes, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    tr_l, va_l, tr_a, va_a, te_a = [], [], [], [], []
    best_va_loss = float('inf')
    best_va, best_te, best_ep = 0.0, 0.0, 0
    bad = 0
    t0 = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = crit(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            tr_l.append(crit(out[data.train_mask], data.y[data.train_mask]).item())
            va_l.append(crit(out[data.val_mask], data.y[data.val_mask]).item())
            tr_a.append(_accuracy(out, data.y, data.train_mask))
            va_a.append(_accuracy(out, data.y, data.val_mask))
            te_a.append(_accuracy(out, data.y, data.test_mask))
        if va_a[-1] > best_va:
            best_va, best_te, best_ep = va_a[-1], te_a[-1], epoch
        if va_l[-1] < best_va_loss:
            best_va_loss = va_l[-1]; bad = 0
        else:
            bad += 1
            if bad >= patience: break
    if device.type == 'cuda': torch.cuda.synchronize()
    return RunResult('GCN', '', seed, best_te, best_va, best_ep,
                     time.perf_counter() - t0, tr_l, va_l, tr_a, va_a, te_a)

## 5. Top-level dispatcher

In [31]:
def run_experiment(model_name, dataset_name, seed, **hparams):
    dataset = Planetoid(root=f'./data/{dataset_name}', name=dataset_name,
                        transform=T.NormalizeFeatures())
    data = dataset[0].to(device)
    if model_name.lower() == 'sgc':
        res = run_sgc(data, dataset.num_classes, seed, **hparams)
    elif model_name.lower() == 'gcn':
        res = run_gcn(data, dataset.num_classes, seed, **hparams)
    else:
        raise ValueError(model_name)
    res.dataset = dataset_name
    return res

## 6. Tune SGC weight decay (per dataset)

Grid-search `weight_decay` over a small log-spaced grid using **validation** accuracy. We use a few seeds per `wd` to reduce noise from random init. The best `wd` per dataset is then used for final test evaluation in the next cell.

In [ ]:
WD_GRID = [1e-7, 1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4]
TUNE_SEEDS = 3
DATASETS = ['Cora', 'Citeseer', 'Pubmed']

def tune_sgc_wd(dataset_name, wd_grid=WD_GRID, tune_seeds=TUNE_SEEDS):
    dataset = Planetoid(root=f'./data/{dataset_name}', name=dataset_name,
                        transform=T.NormalizeFeatures())
    data = dataset[0].to(device)
    grid = {}
    print(f'\nTuning SGC on {dataset_name} (grid={wd_grid}, seeds={tune_seeds}):')
    for wd in wd_grid:
        vas = [run_sgc(data, dataset.num_classes, s, weight_decay=wd).val_acc
               for s in range(tune_seeds)]
        grid[wd] = (np.mean(vas), np.std(vas))
        print(f'  wd={wd:.0e}  val={grid[wd][0]*100:.2f} ± {grid[wd][1]*100:.2f}')
    best = max(wd_grid, key=lambda w: grid[w][0])
    print(f'  → best wd = {best:.0e}')
    return best, grid

SGC_WD = {}
TUNE_GRIDS = {}
for ds in DATASETS:
    SGC_WD[ds], TUNE_GRIDS[ds] = tune_sgc_wd(ds)

print('\n=== Selected weight decays ===')
for ds, wd in SGC_WD.items():
    print(f'  {ds:<10} wd = {wd:.0e}')


Tuning SGC on Cora (grid=[1e-07, 1e-06, 5e-06, 1e-05, 5e-05, 0.0001, 0.0005], seeds=3):
  wd=1e-07  val=77.47 ± 0.09
  wd=1e-06  val=77.80 ± 0.00
  wd=5e-06  val=79.20 ± 0.00
  wd=1e-05  val=79.67 ± 0.09
  wd=5e-05  val=79.53 ± 0.09
  wd=1e-04  val=79.53 ± 0.19
  wd=5e-04  val=78.93 ± 0.66
  → best wd = 1e-05

Tuning SGC on Citeseer (grid=[1e-07, 1e-06, 5e-06, 1e-05, 5e-05, 0.0001, 0.0005], seeds=3):
  wd=1e-07  val=69.40 ± 0.28
  wd=1e-06  val=69.80 ± 0.16
  wd=5e-06  val=70.13 ± 0.09
  wd=1e-05  val=70.40 ± 0.00
  wd=5e-05  val=73.60 ± 0.00
  wd=1e-04  val=73.87 ± 0.09
  wd=5e-04  val=73.53 ± 0.41
  → best wd = 1e-04

Tuning SGC on Pubmed (grid=[1e-07, 1e-06, 5e-06, 1e-05, 5e-05, 0.0001, 0.0005], seeds=3):
  wd=1e-07  val=77.40 ± 0.00
  wd=1e-06  val=77.33 ± 0.09
  wd=5e-06  val=78.00 ± 0.00


In [ ]:
# Visualize the wd grid — useful for the writeup, shows tuning was actually informative
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(DATASETS), figsize=(13, 3.5), sharey=False)
for ax, ds in zip(axes, DATASETS):
    grid = TUNE_GRIDS[ds]
    wds = sorted(grid.keys())
    means = [grid[w][0] * 100 for w in wds]
    stds = [grid[w][1] * 100 for w in wds]
    ax.errorbar(range(len(wds)), means, yerr=stds, marker='o', capsize=4,
                color='#A6192E', linewidth=2)
    best = SGC_WD[ds]
    bi = wds.index(best)
    ax.scatter([bi], [means[bi]], s=180, color='#1E2761', zorder=5,
               label=f'best: {best:.0e}')
    ax.set_xticks(range(len(wds)))
    ax.set_xticklabels([f'{w:.0e}' for w in wds], rotation=30)
    ax.set_xlabel('weight decay')
    ax.set_ylabel('val acc (%)')
    ax.set_title(ds)
    ax.legend(loc='lower center')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Run the full benchmark (with tuned SGC weight decay)

In [ ]:
MODELS = ['sgc', 'gcn']
SEEDS = list(range(10))

results = []
for model_name in MODELS:
    for ds in DATASETS:
        for seed in SEEDS:
            hparams = {}
            if model_name == 'sgc' and ds in SGC_WD:
                hparams['weight_decay'] = SGC_WD[ds]
            r = run_experiment(model_name, ds, seed, **hparams)
            results.append(r)
            print(f'[{model_name:>3}/{ds:<8} seed={seed}] '
                  f'test={r.test_acc*100:.2f}% '
                  f'val={r.val_acc*100:.2f}% '
                  f'epoch={r.best_epoch:3d} '
                  f'time={r.train_time:.3f}s')

## 8. Summary table

In [ ]:
import pandas as pd
rows = []
for model in MODELS:
    for ds in DATASETS:
        sub = [r for r in results if r.model.lower() == model and r.dataset == ds]
        accs = np.array([r.test_acc for r in sub]) * 100
        times = np.array([r.train_time for r in sub])
        rows.append({
            'Model': model.upper(), 'Dataset': ds,
            'Test Acc': f'{accs.mean():.1f} ± {accs.std():.1f}',
            'Train Time (s)': f'{times.mean():.3f} ± {times.std():.3f}',
            'N': len(sub),
        })
df = pd.DataFrame(rows)
df

In [ ]:
# Plot compares training time between SGC and GCN
fig, ax = plt.subplots(figsize=(8, 5))

model_labels = ('SGC', 'GCN')
colors = {'SGC': '#A6192E', 'GCN': '#1E2761'}
x = np.arange(len(DATASETS))
width = 0.35

for i, model in enumerate(model_labels):
  means, stds = [], []
  for ds in DATASETS:
    sub = [r for r in results if r.model.upper() == model and r.dataset == ds]
    times = np.array([r.train_time for r in sub])
    means.append(times.mean())
    stds.append(times.std())
  offset = (i - 0.5) * width
  bars = ax.bar(x + offset, means, width, yerr=stds, label=model, color=colors[model], capsize=5, alpha=0.85, zorder=3)
  # Stats
  for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f'{mean:.3f}s', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(DATASETS)
ax.set_ylabel('Training Time (Seconds)')
ax.set_title('Training Time (SGC vs GCN)')
ax.legend()
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
# Plot compares accuracy in test between SGC and GCN using three different datasets
fig, ax = plt.subplots(figsize=(8, 5))

for i, model in enumerate(model_labels):
  means, stds = [], []
  for ds in DATASETS:
    sub = [r for r in results if r.model.upper() == model and r.dataset == ds]
    accs = np.array([r.test_acc for r in sub]) * 100
    means.append(accs.mean())
    stds.append(accs.std())
  offset = (i - 0.5) * width
  bars = ax.bar(x + offset, means, width, yerr=stds, label=model, color=colors[model], capsize=5, alpha=0.85, zorder=3)
  for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f'{mean:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(DATASETS)
ax.set_ylabel('Test Accuracy (Percentage)')
ax.set_title('Test Accuracy (SGC vs GCN)')
ax.legend()
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

In [ ]:
# Learning Curves for Loss/Accuracy
fig, axes = plt.subplots(2, len(DATASETS), figsize=(14, 8))
for col, ds in enumerate(DATASETS):
  for model, color in colors.items():
    runs = [r for r in results if r.model.upper() == model and r.dataset == ds]

    # Handles ragged lengths via early stopping
    min_len_loss = min(len(r.val_loss_curve) for r in runs)
    min_len_acc = min(len(r.val_acc_curve) for r in runs)

    val_losses = np.array([r.val_loss_curve[:min_len_loss] for r in runs])
    train_losses = np.array([r.train_loss_curve[:min_len_loss] for r in runs])
    val_accs = np.array([r.val_acc_curve[:min_len_acc] for r in runs]) * 100
    train_accs = np.array([r.train_acc_curve[:min_len_acc] for r in runs]) * 100

    # Loss subplot
    ax = axes[0, col]
    for curves, ls in [(train_losses, '--'), (val_losses, '-')]:
      mean, std = curves.mean(axis=0), curves.std(axis=0)
      epochs = np.arange(1, len(mean) + 1)
      ax.plot(epochs, mean, color=color, linestyle=ls, label=f'{model} ({"val" if ls == "-" else "train"})')
      ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.15)
    ax.set_title(ds)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(alpha=0.3)
    if col == 0:
        ax.legend(fontsize=7)

    # Accuracy subplot
    ax = axes[1, col]
    for curves, ls in [(train_accs, '--'), (val_accs, '-')]:
      mean, std = curves.mean(axis=0), curves.std(axis=0)
      epochs = np.arange(1, len(mean) + 1)
      ax.plot(epochs, mean, color=color, linestyle=ls, label=f'{model} ({"val" if ls == "-" else "train"})')
      ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.15)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (Percentage)')
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)

fig.suptitle('Learning Curves (SGC vs GCN)')
plt.tight_layout()
plt.show()

In [ ]:
# Save raw results so step 5 (Figure 3 timing plot) can reuse them
import pickle
with open('results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Saved results.pkl')